# 한 단어씩 읽는 감성 AI 🎬

**한국외국어대학교 인공지능 · RNN 보너스 실습**

RNN은 문장을 한 글자씩 읽으면서 은닉상태 h<sub>t</sub>에 지금까지의 내용을 요약해 다음 시점으로 넘긴다. 이 노트북에서는 네이버 영화 리뷰 20만 건으로 감성 분류 RNN을 만들고, AI가 리뷰를 읽어 가는 동안 판단이 어떻게 바뀌는지 들여다본다.

| 단계 | 할 일 | 시간(T4 GPU) |
|---|---|---|
| 1 | 영화 리뷰 데이터 받기, AI보다 먼저 맞혀 보기 | 1분 |
| 2 | 글자를 숫자로 바꾸기 (음절 단위 토큰화) | 1분 |
| 3 | 기본 RNN과 LSTM 학습, 긴 리뷰에서 누가 더 잘하나 | 2~3분 |
| 4 | **한 단어씩 읽기**: 반전 리뷰에서 판단이 뒤집히는 순간, 은닉상태의 이동 | 2분 |
| 5 | **게임**: AI를 속이는 리뷰 쓰기 | 자유 |
| 6 | 보너스: **AI 리뷰 작가**, 긍정·부정 리뷰를 쓰는 RNN | 3~5분 |

**시작 전에**: `런타임 → 런타임 유형 변경 → T4 GPU`를 고른 뒤 `런타임 → 모두 실행`. API 키는 필요 없다. 그래프의 제목과 축은 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시하고, 리뷰 문장은 그래프 위에 텍스트로 출력한다.

데이터: [Naver Sentiment Movie Corpus (NSMC)](https://github.com/e9t/nsmc), 박은정, CC0 1.0. 네이버 영화 리뷰 20만 건(학습 15만, 평가 5만)에 긍정 1, 부정 0이 붙어 있다.

## 0. 준비

In [ ]:
import importlib.util, subprocess, sys, os, io, math, time, random, urllib.request, collections
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gradio"], check=False)

import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

# 그래프의 제목·축·범례는 Colab에서 한글 글꼴이 깨지지 않도록 영어로 표시한다.
# 리뷰 문장처럼 한글이 필요한 내용은 그래프 위에 텍스트로 출력한다.

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(GPU 없음: 느리지만 동작한다)")

## 1. 영화 리뷰 데이터 받기

In [ ]:
BASE = "https://raw.githubusercontent.com/e9t/nsmc/master/"
for f in ["ratings_train.txt", "ratings_test.txt"]:
    if not os.path.exists(f): urllib.request.urlretrieve(BASE + f, f)
train = pd.read_csv("ratings_train.txt", sep="\t").dropna()
test = pd.read_csv("ratings_test.txt", sep="\t").dropna()
print(f"학습 {len(train):,}건 · 평가 {len(test):,}건 · 긍정 비율 {train.label.mean():.1%}")
print(f"리뷰 길이(글자): 중앙값 {train.document.str.len().median():.0f}, 90% 지점 {train.document.str.len().quantile(.9):.0f}")
train.sample(8, random_state=1)[["document", "label"]]

### 🕵️ 미니 게임: AI보다 먼저 맞혀 보기

아래 10개 리뷰가 긍정(1)인지 부정(0)인지 적어 보자. 다음 셀에서 정답이 나온다. 3단계에서 AI의 점수와 비교한다.

In [ ]:
quiz = test.sample(10, random_state=2026).reset_index(drop=True)
for i, s in enumerate(quiz.document, 1): print(f"Q{i:2d}. {s}")

In [ ]:
for i, (s, y) in enumerate(zip(quiz.document, quiz.label), 1): print(f"Q{i:2d}. {'긍정' if y else '부정'}  {s}")

> 🤔 정답이 이상해 보이는 리뷰가 있는가? 평점과 글의 내용이 어긋나는 리뷰도 데이터에 섞여 있다. 사람도 100%를 맞히기 어렵다는 점을 기억해 두자.

## 2. 글자를 숫자로 바꾸기

RNN에는 숫자를 넣어야 한다. 여기서는 **음절(글자) 하나를 한 시점**으로 본다. 한국어는 "연기는", "연기가", "연기도"처럼 단어에 조사가 붙어 모양이 계속 바뀌는데, 음절 단위로 자르면 약 2천 개의 글자만으로 모든 리뷰를 표현할 수 있다.

In [ ]:
cnt = collections.Counter(ch for s in train.document for ch in s)
vocab = ["<pad>", "<unk>"] + [c for c, n in cnt.most_common() if n >= 3 and (c == " " or c.isprintable() and not c.isspace())]
stoi = {c: i for i, c in enumerate(vocab)}
MAXLEN = 100
print("글자 사전 크기:", len(vocab))

def encode(texts, maxlen=MAXLEN):
    X = np.zeros((len(texts), maxlen), np.int64); n = np.ones(len(texts), np.int64)
    for i, s in enumerate(texts):
        ids = [stoi.get(c, 1) for c in s[:maxlen]]
        X[i, :len(ids)] = ids; n[i] = max(len(ids), 1)
    return torch.tensor(X), torch.tensor(n)

ex = "연기는 좋았는데 별로"
x, n = encode([ex])
print(ex, "→", x[0, :n[0]].tolist())
Xtr, ntr = encode(train.document.tolist()); ytr = torch.tensor(train.label.values)
Xte, nte = encode(test.document.tolist());  yte = torch.tensor(test.label.values)

## 3. 기본 RNN과 LSTM 학습

같은 구조에서 순환층만 바꿔 두 모델을 학습한다.

```
글자 번호 → 임베딩(64) → 순환층(RNN 또는 LSTM, 은닉 128) → 마지막 은닉상태 h_T → 선형층 → 긍정 확률
```

리뷰를 끝까지 읽은 뒤 **마지막 은닉상태 하나로 판정하는 다대일** 형태다. 강의자료 M7의 「RNN의 입력과 출력 형태」 카드에서 본 구조다.

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, cell="LSTM", emb=64, hid=128):
        super().__init__()
        self.emb = nn.Embedding(len(vocab), emb, padding_idx=0)
        self.rnn = getattr(nn, cell)(emb, hid, batch_first=True)
        self.head = nn.Linear(hid, 1)
    def forward(self, x, n):
        out, _ = self.rnn(self.emb(x))                    # out: 모든 시점의 h_t
        h_last = out[torch.arange(len(x)), n - 1]         # 리뷰의 마지막 글자 위치의 h_T
        return self.head(h_last).squeeze(1), out

@torch.no_grad()
def evaluate(model):
    model.eval(); logits = []
    for i in range(0, len(Xte), 2048):
        logits.append(model(Xte[i:i+2048].to(device), nte[i:i+2048].to(device))[0].cpu())
    return ((torch.cat(logits) > 0).long() == yte).float()

def train_model(cell, epochs=3, bs=256):
    torch.manual_seed(SEED)
    m = SentimentRNN(cell).to(device); opt = torch.optim.Adam(m.parameters(), 2e-3); t0 = time.time()
    for ep in range(1, epochs + 1):
        m.train(); order = torch.randperm(len(Xtr))
        for i in range(0, len(order), bs):
            b = order[i:i+bs]
            logit, _ = m(Xtr[b].to(device), ntr[b].to(device))
            loss = F.binary_cross_entropy_with_logits(logit, ytr[b].float().to(device))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        print(f"{cell:4s} epoch {ep}: 평가 정확도 {evaluate(m).mean():.1%} · {time.time()-t0:.0f}초")
    return m

lstm = train_model("LSTM")
rnn = train_model("RNN")

리뷰 길이별로 정확도를 나눠 보자. 강의에서 본 기울기 소실 때문에, 기본 RNN은 긴 리뷰의 앞부분 정보를 잘 전달하지 못할 수 있다.

In [ ]:
lens = nte.numpy()
bins = [(1, 20, "<=20 chars"), (21, 59, "21-59 chars"), (60, 999, ">=60 chars")]
acc_l, acc_r = evaluate(lstm), evaluate(rnn)
print(f"{'길이':14s}{'리뷰 수':>9s}{'기본 RNN':>10s}{'LSTM':>8s}")
rows = []
for a, b, name in bins:
    m = (lens >= a) & (lens <= b)
    rows.append((name, acc_r[m].mean().item(), acc_l[m].mean().item()))
    print(f"{name:14s}{m.sum():>9,}{rows[-1][1]:>10.1%}{rows[-1][2]:>8.1%}")
print(f"{'전체':14s}{len(lens):>9,}{acc_r.mean():>10.1%}{acc_l.mean():>8.1%}")

x = np.arange(len(rows))
plt.figure(figsize=(6, 3.2))
plt.bar(x - 0.2, [r[1] for r in rows], 0.4, label="Basic RNN", color="#A6432E")
plt.bar(x + 0.2, [r[2] for r in rows], 0.4, label="LSTM", color="#146E7A")
plt.xticks(x, [r[0] for r in rows]); plt.ylim(0.7, 0.9); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy by review length")
plt.tight_layout(); plt.show()

p = torch.sigmoid(lstm(*[t.to(device) for t in encode(quiz.document.tolist())])[0]).detach().cpu().numpy()
print(f"\n미니 게임 10문제 · LSTM 점수: {((p > 0.5).astype(int) == quiz.label.values).sum()} / 10")

## 4. 한 단어씩 읽기 👀

RNN은 매 글자마다 은닉상태 h<sub>t</sub>를 만든다. 학습할 때는 마지막 h<sub>T</sub>만 썼지만, 중간의 h<sub>t</sub>에도 같은 선형층을 붙이면 **"리뷰가 여기서 끝났다면 AI는 어떻게 판단했을까"**를 볼 수 있다. 각 단어(어절)의 마지막 글자에서 긍정 확률을 읽는다.

In [ ]:
@torch.no_grad()
def read_along(model, text):
    # 어절마다 (단어, 그 단어까지 읽었을 때의 긍정 확률)을 돌려준다.
    model.eval()
    x, n = encode([text]); _, out = model(x.to(device), n.to(device))
    probs = torch.sigmoid(model.head(out[0, :n[0]])).squeeze(1).cpu().numpy()
    words, res, pos = text.split(" "), [], 0
    for w in words:
        if not w: pos += 1; continue
        pos += len(w)
        res.append((w, float(probs[min(pos, len(probs)) - 1]))); pos += 1
    return res

COLORS = {"LSTM": "#146E7A", "기본 RNN": "#A6432E"}
EN = {"LSTM": "LSTM", "기본 RNN": "Basic RNN"}

def plot_read(text, models):
    # 단어(어절)는 그래프에 w1, w2 … 번호로 표시하고, 번호와 단어는 위에 텍스트로 출력한다.
    words = [w for w in text.split(" ") if w]
    print(f"리뷰: {text}")
    print("   " + "   ".join(f"w{i+1}={w}" for i, w in enumerate(words)))
    plt.figure(figsize=(max(5, 1.1 * len(words)), 3))
    for name, m in models:
        r = read_along(m, text)
        plt.plot(range(len(r)), [p for _, p in r], "o-", label=EN.get(name, name), lw=2, color=COLORS.get(name))
    plt.axhline(0.5, color="#B7C4C8", ls="--"); plt.ylim(-0.05, 1.05)
    plt.xticks(range(len(r)), [f"w{i+1}" for i in range(len(r))]); plt.ylabel("P(positive)"); plt.xlabel("word read so far")
    plt.title("Positive probability after each word", fontsize=11)
    plt.legend(); plt.tight_layout(); plt.show()

TRICKY = [
    "재미없을 줄 알았는데 진짜 재밌다",
    "연기는 정말 좋았는데 스토리가 너무 별로였다",
    "시간 가는 줄 모르고 봤어요",
    "이걸 돈 주고 봤다니",
    "최고의 영화 인생작입니다",
]
for t in TRICKY:
    plot_read(t, (("LSTM", lstm), ("기본 RNN", rnn)))

> 🤔 "재미없을 줄 알았는데 진짜 재밌다"에서 판단이 언제 뒤집히는가? "이걸 돈 주고 봤다니"처럼 부정적인 단어가 없는 리뷰는 왜 부정으로 읽을까? 학습 데이터에 비슷한 표현이 많았을 가능성을 생각해 보자.

### 은닉상태는 어떻게 움직일까

LSTM의 은닉상태 h<sub>t</sub>는 128개 숫자다. 리뷰를 읽는 동안 이 128개 숫자가 어떻게 바뀌는지 열지도로 보고, 2차원으로 줄여(PCA) 이동 경로도 그린다. 긍정 리뷰와 부정 리뷰의 끝 지점을 비교해 보자. 그래프의 글자 번호와 리뷰 기호(A, B, C)는 그래프 위에 출력된 목록과 대응한다.

In [ ]:
@torch.no_grad()
def hidden_states(model, text):
    x, n = encode([text]); _, out = model(x.to(device), n.to(device))
    return out[0, :n[0]].cpu().numpy()

text = TRICKY[0]
H = hidden_states(lstm, text)
plt.figure(figsize=(11, 3.2))
plt.imshow(H.T[:48], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
print(f"리뷰: {text}")
print("   " + " ".join(f"{i+1}:{c}" for i, c in enumerate(text)))
plt.xticks(range(len(text)), [str(i + 1) for i in range(len(text))]); plt.xlabel("time step t (character)"); plt.ylabel("hidden unit (first 48)")
plt.title("Hidden state h_t at each character"); plt.colorbar(fraction=0.02); plt.tight_layout(); plt.show()

# 여러 리뷰의 은닉상태를 모아 PCA로 2차원 평면을 만들고, 리뷰 하나의 이동 경로를 그린다.
sample = test.sample(400, random_state=0).document.tolist()
pool = np.concatenate([hidden_states(lstm, s) for s in sample])
mu = pool.mean(0); U, S, Vt = np.linalg.svd(pool - mu, full_matrices=False); P = Vt[:2].T
plt.figure(figsize=(6.5, 5))
for tag, t in zip("ABC", [TRICKY[0], TRICKY[1], TRICKY[4]]): print(f"review {tag}: {t}")
for tag, t, col in [("A", TRICKY[0], "#146E7A"), ("B", TRICKY[1], "#A6432E"), ("C", TRICKY[4], "#002B49")]:
    Z = (hidden_states(lstm, t) - mu) @ P
    plt.plot(Z[:, 0], Z[:, 1], "-", color=col, alpha=0.6); plt.scatter(Z[-1, 0], Z[-1, 1], s=90, color=col, label=f"review {tag}")
    plt.scatter(Z[0, 0], Z[0, 1], s=25, color=col, marker="x")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.title("Path of the hidden state (x: start, dot: end)")
plt.legend(fontsize=9, loc="best"); plt.tight_layout(); plt.show()

## 5. 🎮 게임: AI를 속이는 리뷰 쓰기

아래 셀을 실행하면 입력창이 나온다. 리뷰를 쓰면 단어마다 색이 칠해진다. **파랑은 긍정, 빨강은 부정** 쪽으로 판단이 기울었다는 뜻이고, 진할수록 확신이 크다.

- 사람이 읽으면 분명히 긍정인데 AI는 부정으로 읽는 리뷰(또는 그 반대)를 만들어 보자.
- 반어법("참 잘~ 만들었네"), 부정("나쁘지 않다"), 반전("~줄 알았는데"), 신조어, 이모티콘을 써 본다.
- 가장 크게 속인 리뷰를 팀별로 뽑아 발표한다.

In [ ]:
def game(text, model_name):
    text = (text or "").strip()
    if not text: return [], {}
    m = lstm if model_name == "LSTM" else rnn
    r = read_along(m, text)
    colored = []
    for w, p in r:
        colored.append((w + " ", round(2 * p - 1, 3)))          # -1(부정) ~ +1(긍정)
    final = r[-1][1]
    return colored, {"긍정": final, "부정": 1 - final}

import gradio as gr
with gr.Blocks(title="한 단어씩 읽는 감성 AI") as app:
    gr.Markdown("### 한 단어씩 읽는 감성 AI: 리뷰를 쓰면 단어마다 AI의 판단이 색으로 보인다 (파랑=긍정, 빨강=부정)")
    with gr.Row():
        with gr.Column():
            box = gr.Textbox(label="영화 리뷰", value="재미없을 줄 알았는데 진짜 재밌다", lines=2)
            which = gr.Radio(["LSTM", "기본 RNN"], value="LSTM", label="모델")
        with gr.Column():
            hl = gr.HighlightedText(label="단어까지 읽었을 때의 판단", combine_adjacent=False, show_legend=False)
            lab = gr.Label(label="최종 판정")
    box.change(game, [box, which], [hl, lab]); which.change(game, [box, which], [hl, lab])
    app.load(game, [box, which], [hl, lab])
app.launch(debug=False)

## 6. 보너스: AI 리뷰 작가 ✍️

지금까지는 리뷰를 **읽는** RNN이었다. 이번에는 리뷰를 **쓰는** RNN이다. 학습 방법은 1990년 Elman의 실험부터 지금의 GPT까지 이어지는 **다음 글자 예측**이다. 문장 맨 앞에 `<긍정>` 또는 `<부정>` 표시를 붙여 학습하면, 생성할 때 이 표시로 분위기를 고를 수 있다.

```
입력:  <긍정>  인 생 영 화 다 !
정답:   인 생 영 화 다 ! <끝>
```

In [ ]:
gen_df = train[train.document.str.len().between(8, 60)]
gcnt = collections.Counter(ch for s in gen_df.document for ch in s)
gvocab = ["<pad>", "<부정>", "<긍정>", "<끝>", "<unk>"] + [c for c, k in gcnt.most_common() if k >= 5 and (c == " " or c.isprintable() and not c.isspace())]
gstoi = {c: i for i, c in enumerate(gvocab)}
GL = 62
G = np.zeros((len(gen_df), GL), np.int64)
for i, (s, y) in enumerate(zip(gen_df.document, gen_df.label)):
    ids = [1 + y] + [gstoi.get(c, 4) for c in s] + [3]; G[i, :len(ids)] = ids
G = torch.tensor(G, device=device)

class Writer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(len(gvocab), 128)
        self.rnn = nn.LSTM(128, 384, num_layers=2, batch_first=True, dropout=0.1)
        self.out = nn.Linear(384, len(gvocab))
    def forward(self, x, h=None):
        o, h = self.rnn(self.emb(x), h); return self.out(o), h

torch.manual_seed(SEED)
writer = Writer().to(device); opt = torch.optim.Adam(writer.parameters(), 2e-3)
STEPS = 3000 if device == "cuda" else 400
t0 = time.time()
for step in range(1, STEPS + 1):
    b = G[torch.randint(0, len(G), (256,), device=device)]
    logits, _ = writer(b[:, :-1])                       # 시점마다 다음 글자 예측 (다대다)
    loss = F.cross_entropy(logits.reshape(-1, len(gvocab)), b[:, 1:].reshape(-1), ignore_index=0)
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(writer.parameters(), 1.0); opt.step()
    if step % 500 == 0 or step == STEPS:
        print(f"step {step:5d} · 다음 글자 예측 손실 {loss.item():.3f} · {time.time()-t0:.0f}초")

In [ ]:
@torch.no_grad()
def write(mood="긍정", start="", temp=0.8, n=5, seed=None):
    # mood: "긍정" 또는 "부정", start: 첫 글자들(선택), temp: 온도
    writer.eval(); g = torch.Generator(device=device)
    if seed is not None: g.manual_seed(seed)
    outs = []
    for _ in range(n):
        ids = [gstoi[f"<{mood}>"]] + [gstoi.get(c, 4) for c in start]
        x = torch.tensor([ids], device=device); h = None; text = start
        for _ in range(60):
            logits, h = writer(x, h)
            p = F.softmax(logits[0, -1] / temp, 0)
            k = torch.multinomial(p, 1, generator=g).item()
            if k == 3: break
            text += gvocab[k] if k > 4 else ""; x = torch.tensor([[k]], device=device)
        outs.append(text)
    return outs

for mood in ["긍정", "부정"]:
    for temp in [0.5, 1.0, 1.5]:
        print(f"\n[{mood} · 온도 {temp}]")
        for s in write(mood, temp=temp, n=3, seed=1): print("  ", s)

> 🤔 온도를 0.5에서 1.5로 올리면 문장이 어떻게 바뀌는가? 온도는 다음 글자의 확률분포를 뾰족하게(낮을 때) 또는 평평하게(높을 때) 만든다. LLM의 temperature 설정과 같은 원리다.

AI 작가가 쓴 리뷰를 3단계의 감성 AI(LSTM)에게 채점시켜 보자. 작가가 "긍정"으로 쓴 리뷰를 감성 AI도 긍정으로 읽을까?

In [ ]:
for mood in ["긍정", "부정"]:
    texts = write(mood, temp=0.8, n=40, seed=7)
    p = torch.sigmoid(lstm(*[t.to(device) for t in encode(texts)])[0]).detach().cpu().numpy()
    hit = (p > 0.5).mean() if mood == "긍정" else (p < 0.5).mean()
    print(f"작가가 '{mood}'으로 쓴 40개 중 감성 AI가 같은 쪽으로 읽은 비율: {hit:.0%}")

print("\n첫 글자를 정해 주고 이어 쓰게 하기:")
for s in write("긍정", start="이 영화는", temp=0.8, n=3, seed=3): print("  긍정:", s)
for s in write("부정", start="이 영화는", temp=0.8, n=3, seed=3): print("  부정:", s)

## 7. 도전 과제

1. **AI 속이기 대회**: 5단계 게임에서 사람은 긍정으로 읽는데 AI는 부정으로 읽는 리뷰를 3개 찾는다. 어떤 표현이 AI를 헷갈리게 했는지 단어별 색으로 설명한다.
2. **RNN 대 LSTM**: 3단계의 epochs를 5로 늘려도 긴 리뷰에서 기본 RNN과 LSTM의 차이가 유지되는가?
3. **양방향 읽기**: `SentimentRNN`의 순환층에 `bidirectional=True`를 넣고 선형층 입력을 256으로 바꿔 정확도를 비교한다. 4단계의 "단어까지 읽은 판단"은 양방향 모델에서도 같은 의미일까?
4. **작가 길들이기**: `write("부정", start="배우들 연기는")`처럼 시작 문장을 바꿔 가며, 긍정 시작 문장 + 부정 분위기처럼 서로 어긋나는 조건에서 무엇이 나오는지 본다.
5. **작명소로 바꾸기**: 6단계의 리뷰 데이터 대신 원하는 이름 목록(카페 이름, 동네 이름 등)을 한 줄에 하나씩 넣어 학습하면 새 이름을 짓는 RNN이 된다.

*데이터: Naver Sentiment Movie Corpus (NSMC), Lucy Park, CC0 1.0. AI 작가가 만든 문장은 학습 데이터의 패턴을 따라 만든 것이며 실제 리뷰가 아니다.*